<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

> **Prof. Heider Sanchez**  

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [36]:
import psycopg2
import pandas as pd
import json

def connect_db():
    conn = psycopg2.connect(
        dbname="Lab08",
        user="postgres",
        password="postgres",
        host="localhost",
        port = 5432
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    #df['bag_of_words'] = df['bag_of_words'].apply(json.loads)
    conn.close()
    return df

noticias_df = fetch_data()
display(noticias_df.head())

C:\Users\Familia\AppData\Local\Temp\ipykernel_11400\4021732495.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,contenido,bag_of_words
0,1,Durante el foro La banca articulador empresari...,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,2,El regulador de valores de China dijo el domin...,"{'6': 1, 'dat': 1, 'deb': 1, 'inc': 1, 'med': ..."
2,3,En una industria históricamente masculina como...,"{'15': 1, '20': 1, '21': 1, '42': 1, '50': 1, ..."
3,4,Con el dato de marzo el IPC interanual encaden...,"{'3': 1, '9': 1, '07': 1, '09': 1, '11': 2, '2..."
4,5,Ayer en Cartagena se dio inicio a la versión n...,"{'17': 1, '23': 1, '31': 1, '34': 1, '50': 2, ..."


## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [37]:
import math


class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}

    def build_from_db(self):
        # Leer desde PostgreSQL todos los bag of words
        # Construir el índice invertido, el idf y la norma (longitud) de cada documento
        
        global noticias_df
        N = len(noticias_df)

        #indice invertido
        for _, row in noticias_df.iterrows():
            doc_id = row['id']
            bow= row['bag_of_words']

            for word, tf in bow.items():
                if word not in self.index:
                    self.index[word] = []
                #se guarda tupla (doc_id, frecuencia d termino) para cada palabra
                self.index[word].append((doc_id, tf))


        #calcular idf para cada palabra
        for word, posting_list in self.index.items():
            df = len(posting_list)  #cantidad de docs q tienen esa palabra
            self.idf[word] = math.log10(N / df) if df > 0 else 0
        
        
        #calcular longitud de cada documento
        for _, row in noticias_df.iterrows():
            doc_id = row['id']
            bow = row['bag_of_words']
            
            suma_cuadrados = 0
            for word, tf in bow.items():
                peso_tfidf= tf*self.idf[word]
                suma_cuadrados+= peso_tfidf** 2
            self.length[doc_id]= math.sqrt(suma_cuadrados)
        
        print(f"Vocabulario total: {len(self.index)} palabras")

    
    def L(self, word):
        # Retorna la lista de documentos que contienen a word 
        return self.index.get(word, [])

In [38]:
idx = InvertedIndex()
idx.build_from_db()

Vocabulario total: 14457 palabras


## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [39]:
import nltk
from nltk.stem.snowball import SnowballStemmer

for word in idx.index:
    idx.index[word] = sorted(idx.index[word], key=lambda x: x[0])

def AND(list1, list2):
    result = []
    i, j = 0, 0
    
    while i < len(list1) and j < len(list2):
        doc_id1 = list1[i][0]
        doc_id2 = list2[j][0]
        
        if doc_id1 == doc_id2:
            result.append(list1[i])
            i += 1
            j += 1
        elif doc_id1 < doc_id2:
            i += 1
        else:
            j += 1
            
    return result

def OR(list1, list2):
    result = []
    i, j = 0, 0
    
    while i < len(list1) and j < len(list2):
        doc_id1 = list1[i][0]
        doc_id2 = list2[j][0]
        
        if doc_id1 == doc_id2:
            result.append(list1[i])
            i += 1
            j += 1
        elif doc_id1 < doc_id2:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            j += 1
            
    while i < len(list1):
        result.append(list1[i])
        i += 1
    while j < len(list2):
        result.append(list2[j])
        j += 1
        
    return result

def AND_NOT(list1, list2):
    result = []
    i, j = 0, 0
    
    while i < len(list1) and j < len(list2):
        doc_id1 = list1[i][0]
        doc_id2 = list2[j][0]
        
        if doc_id1 == doc_id2:
            i += 1
            j += 1
        elif doc_id1 < doc_id2:
            result.append(list1[i])
            i += 1
        else:
            j += 1
            
    while i < len(list1):
        result.append(list1[i])
        i += 1
        
    return result


stemmer = SnowballStemmer("spanish")

def buscar(palabra):
    palabra_procesada = stemmer.stem(palabra.lower())
    return idx.L(palabra_procesada)


result = AND(buscar("sostenibilidad"), AND(buscar("ambiente"), buscar("renovables")))
print("sostenibilidad AND ambiente AND renovables: ", [doc[0] for doc in result])

result = AND(buscar("tecnología"), OR(buscar("banca"), buscar("finanzas")))
print("tecnología AND (banca OR finanzas): ", [doc[0] for doc in result])

result = AND_NOT(buscar("economía"), buscar("inflación"))
print("economía AND-NOT inflación: ", [doc[0] for doc in result])

result_extra4 = AND_NOT(AND(buscar("banca"), buscar("finanzas")), buscar("tecnología"))
print("(banca AND finanzas) AND-NOT tecnología: ", [doc[0] for doc in result_extra4])

result_extra5 = AND(buscar("ambiente"), OR(buscar("sostenibilidad"), buscar("economía")))
print("ambiente AND (sostenibilidad OR economía): ", [doc[0] for doc in result_extra5])

sostenibilidad AND ambiente AND renovables:  [52, 397, 420, 469, 637, 1126, 1165, 1179]
tecnología AND (banca OR finanzas):  [12, 13, 15, 16, 17, 20, 30, 49, 52, 55, 63, 78, 85, 88, 93, 111, 116, 135, 144, 147, 156, 178, 179, 187, 193, 194, 205, 206, 210, 212, 226, 228, 234, 237, 246, 253, 258, 259, 269, 282, 286, 293, 302, 312, 321, 332, 358, 362, 365, 369, 376, 391, 394, 406, 415, 416, 417, 419, 430, 445, 455, 456, 460, 465, 478, 498, 507, 511, 522, 540, 543, 571, 573, 583, 584, 594, 598, 601, 606, 607, 611, 617, 623, 626, 628, 633, 639, 642, 644, 655, 665, 669, 674, 675, 679, 686, 689, 696, 705, 719, 723, 738, 748, 781, 786, 793, 819, 826, 829, 840, 842, 849, 850, 852, 864, 888, 889, 892, 895, 901, 905, 911, 916, 917, 918, 943, 951, 958, 979, 988, 992, 996, 1009, 1011, 1015, 1023, 1024, 1033, 1034, 1035, 1042, 1048, 1050, 1052, 1056, 1062, 1065, 1067, 1069, 1079, 1094, 1095, 1096, 1097, 1108, 1113, 1115, 1117, 1121, 1126, 1130, 1136, 1144, 1145, 1149, 1158, 1163, 1164, 1169, 1175, 1

## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

In [40]:
import re

def cosine_search(self, query, top_k=5):  
    score = {}
    stemmer = SnowballStemmer("spanish")
    
    # c procesamos la query
    tokens = re.findall(r'\b\w+\b', query.lower()) 
    query_tf = {}
    for token in tokens:
        word = stemmer.stem(token)
        query_tf[word] = query_tf.get(word, 0) + 1
    
    # c norma
    query_length = 0
    for word, tf in query_tf.items():
        if word in self.idf:
            query_length += (tf * self.idf[word]) ** 2
    query_length = math.sqrt(query_length) if query_length > 0 else 1.0

    # cosineScore
    for word, tf_q in query_tf.items():
        if word not in self.idf:
            continue

        w_tq = tf_q * self.idf[word]
        posting_list = self.L(word)
        
        for doc_id, tf_td in posting_list:
            
            w_td = tf_td * self.idf[word]
            
            score[doc_id] = score.get(doc_id, 0.0) + (w_td * w_tq)

    for doc_id in score:
        score[doc_id] = score[doc_id] / (self.length[doc_id] * query_length)
    
    result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
    
    return result[:top_k]

####  Pruebas funcionales

In [41]:
def showDocument(self, doc_id):
    # Busca el id en el dataframe y retorna los primeros 150 caracteres
    contenido = noticias_df[noticias_df['id'] == doc_id]['contenido'].values
    if len(contenido) > 0:
        return contenido[0][:150] + "..."
    return "Documento no encontrado"

In [42]:
InvertedIndex.cosine_search = cosine_search
InvertedIndex.showDocument = showDocument

In [43]:
test_queries = [
    {
        "query": "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?",
        "top_k": 2
    },
    {
        "query": "evolución de la inflación y el crecimiento de la economía en los últimos años",
        "top_k": 2
    },
    {
        "query": "avances sobre sostenibilidad y energías renovables para el medio ambiente",
        "top_k": 2
    },
    {
        "query": "impacto del conlictos internacionales en la economía global",
        "top_k": 2
    },
    {
        "query": "El desarrollo de videojuegos con el uso de herramientas de software libre",
        "top_k": 2
    },
    {
        "query": "la producción de peliculas em latinoamerica y el futuro de la industria",
        "top_k": 2
    },
    {
        "query": "La administración de recursos naturales en la minería y el impacto ambiental",
        "top_k": 2
    },
    {
        "query": "¿Cuáles son las tendecias actuales en la inversión en energías renovables y su impacto en el mercado financiero?",
        "top_k": 2
    },
    {
        "query": "La animación moderna y la combinación de técnicas tradicionales y digitales en la industria del cine",
        "top_k": 2
    }

]

for test in test_queries:    
    results = idx.cosine_search(test['query'], test['top_k'])
    print(f"Top {test['top_k']} documentos más similares de la query {test_queries.index(test) + 1}:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}: {score:.3f}: ", idx.showDocument(doc_id))

Top 2 documentos más similares de la query 1:
Doc 259: 0.132:  Internet ha sido quizás la innovación tecnológica más grande de los últimos tiempos. Revolucionaria, disruptiva, básica o sostenida, la innovación bus...
Doc 390: 0.118:  La revista Global Finance dio a conocer sus galardones 2021 Best Digital Bank Awards. Bbva fue el único banco español reconocido por la publicación en...
Top 2 documentos más similares de la query 2:
Doc 806: 0.080:  Un alza de 100 puntos base pb es lo que esperan los operadores financieros que aplique el Banco Central a la Tasa de Política Monetaria en su reunión ...
Doc 856: 0.062:  Sobre el mediodía el Dane revelará el dato de cuánto aumentó el precio de vida de los colombianos en octubre y le dará o no la razón a los analistas y...
Top 2 documentos más similares de la query 3:
Doc 523: 0.202:  A pesar de que contribuyen con la sostenibilidad, las energías limpias y las energías renovables no son lo mismo. Las primeras serían aquellas fuentes...
Doc 150